# 12. Sensitivity analysis

## Numerical experiments - Week 45/2025

_Boyan Mihaylov, MSc Computational Science (UVA/VU)_

The procedure presented below performs Bayesian Model Averaging (BMA) on the broad collection of models to
- assess their likelihood of representing the data (with its inherent variance),
- penalize overly complex models,
- assess the models' predictive capabilities
- and explore the combined predictive power of weighted combinations of models.

The aim is to obtain a list of models ranked by their pseudo-BMA weights, which evaluate predictive accuracy under uncertainty and are thus a better criterion than RMSE of model fits to data.

## Prerequisite libraries

In [2]:
using PyPlot
using Revise

# include("../src/conversions.jl")
# include("../src/diffusion.jl")
# include("../src/setup.jl")
# include("../src/plotting.jl")
# include("../src/analysis.jl")
# include("../src/datautils.jl")
# include("../src/germstats.jl")

Revise.includet("../src/conversions.jl")
Revise.includet("../src/diffusion.jl")
Revise.includet("../src/setup.jl")
Revise.includet("../src/plotting.jl")
Revise.includet("../src/analysis.jl")
Revise.includet("../src/datautils.jl")
Revise.includet("../src/germstats.jl")

using .Conversions
using .Diffusion
using .Setup
using .Plotting
using .Analysis
using .DataUtils
using .GermStats

## 1. Parameter priors

The first step is to assume prior distributions of the model parameters, which encode the expectations of plausible parameter values before relating them to data.

### 1.1. Parameter summary

The free parameters in the germination models are:

- $P_s^\textrm{I}$ - the permeation coefficient of the inhibitor molecule through the cell wall;
- $P_s^\textrm{C}$ - the permeation coefficient of the inducer molecule (carbon source) through the cell wall;
- $K_\textrm{I}$ - the half-saturation constant of the inhibitor when directly blocking the inducing signal;
- $K_T^\textrm{I}$ - the half-saturation constant of the inhibitor when shifting the induction threshold;
- $K_T^\textrm{C}$ - the half-saturation constant of the inducer when shifting the inhibition threshold;
- $s$ - scaling factor of the permeability perturbation caused by the inducing signal;
- $b$ - scaling factor of the permeability perturbation caused by the inhibitory signal;
- $k_\textrm{I}$ - scaling factor of the induction threshold shift caused by the inhibitory signal;
- $k_\textrm{C}$ - scaling factor of the inhibition threshold shift caused by the inducing signal;
- $n$ - Hill exponent of the direct inhibition of the inducing signal;
- $\mu_\gamma$ - mean of the inhibition threshold;
- $\sigma_\gamma$ - standard deviation of the inhbition threshold;
- $\mu_\omega$ - mean of the induction threshold;
- $\sigma_\omega$ - standard deviation of the induction threshold;
- $\mu_\psi$ - mean of the initial inhibitor concentration in the spore;
- $\sigma_\psi$ - standard deviation of the initial inhibitor concentration in the spore.

Some of these parameters are independent of the type of carbon source, others are inducer-specific and therefore get a specific instance (with its own distribution) in each inducer case. The inducer-specific parameters are $P_s^\textrm{C}$, $K_\textrm{I}$, $K_T^\textrm{I}$, $K_T^\textrm{C}$, $s$, $k_\textrm{I}$, $k_\textrm{C}$, $n$, $\mu_\omega$, $\sigma_\omega$. On the other hand, $P_s^\textrm{I}$, $b$, $\mu_\gamma$, $\sigma_\gamma$, $\mu_\psi$ and $\sigma_\psi$ are general for all inducer cases.

### 1.2. Determining the priors

Since an exact quantification of the prior distributions is generally difficult to infer from literature, several approaches can be employed to narrow down the bounds, shape an scale of each parameter prior:

1. Use biologically plausible bounds and shapes wherever known.
2. If not known, use log-normal distributions within generously wide bounds, such that 95% of the distribution fall within the bounds.
3. Sample the parameter space, fit the Dantigny triplets to each parameter sample and inspect the distributions of the Dantigny triplets. If a large discrepancy is noticeable compared to recorded data (certain values of a parameter consistently yield unplausible $p_{\textrm{max}}$, $\tau_g$ or $\nu$), narrow down the bounds accordingly. (Prior predictive checks)

#### 1.2.1. Initial guesses

It has been shown that to achieve realistic inhibitor depletion within 4 hours, $P_s^\textrm{I}$ needs to be in the order of $10^{-9}$ to $10^{-7}$. However, inference from physical properties points at a permeability closer to the range of $10^{-5}$ to $10^{-3}$. If strong adsorption is at hand, the former permeability range can be taken, arguing that the adsorption effect is implicit in the low permeation values. However, one could also argue that the permeability is as high as the physics suggest, but it is some secondary interactions that slow down the permeation trigger. Therefore, the high-permeability range is to be considered as well. This results in the liberal bounds of $10^{-9}$ to $10^{-3}$. It should be noted that in some models the permeation coefficient is taken as a base for a temporally modulated shift and in others it remains constant.

As for $P_s^\textrm{C}$, it can remain relatively high if the hydrophobin layer is barely limiting ($10^{-3}$) or extremely low if the full blocking potential of a hydrophobin layer is considered ($10^{-11}$). The first case is conceivable e.g. when an inhibitor-dependent permeability shift is removed, analogous to a deconstruction of the outer cell wall. Thus, a range of $10^{-11}$ to $10^{-3}$ can be assumed.

$\mu_\psi$ has previously been estimated around $10^{-5} \ \textrm{M}$ based on experimental observations of excreted 1-octen-3-ol. It is possible that a large portion of 1-octen-3-ol remained bound to the polysaccharide matrix upon measurement - an upper bound of $10^{-3} \ \textrm{M}$ could accommodate such a possibility.

$K_\textrm{I}$, $K_T^\textrm{I}$ and $K_T^\textrm{C}$ can all be related to typical molecular concentrations of the inhibitor and the inducer inside the spore. For the respective Michaelis-Menten-like relation, it can be said the effect becomes relatively negligible when the half-saturation constant is more than a thousand times larger than a typical concentration. Based on $\mu_\psi$, this amounts to an upper extreme of $1\ \textrm{M}$. The characteristic internal inducer concentration, on the other hand, is highly dependent on the cell wall permeability and the external availability of inducing molecules. In the case of fast equilibration of the currently considered external concentration of $0.01\ \textrm{mM}$, the upper bound for the inducer half-concentration could be set to $10\ \textrm{M}$. Conversely, an extremely low half-saturation would imply extreme sensitivity to molecular concentrations. A lower bound of $10^{-9}\ \textrm{M}$ for all half-saturations reasonably links the visible effects to merely hundreds of molecules.

In [10]:
A, V = compute_spore_area_and_volume_from_dia(5)
inducer_concentration(0.01, 14400, cm_to_um(1e-11), A, compute_ps_layer_volume(2.5, 0.01, 0.2))

0.0003227363405841799